# Preprocessing and Anonymization

This notebook is the only public script that needs access to the original Bright Data snapshots. It first generates the aggregated MSA-level firm-address counts using the original address fields, then produces three row-level anonymized files containing only `ID`, `zip_code`, `mailing_zip`, `city`, `state`, `specializations`, and `number_of_specializations`.

**Do not upload the original Bright Data snapshots.**

In [1]:
# ============================================================
# BRIGHTDATA PREPROCESSING AND ANONYMIZATION
#
# This notebook is the only step that reads the original
# BrightData snapshots containing personally identifying
# information.
#
# It writes:
#   1. firm_address_counts_by_MSA.csv
#   2. BrightData_Lawyers_Anonymized_1.csv
#   3. BrightData_Lawyers_Anonymized_2.csv
#   4. BrightData_Lawyers_Anonymized_3.csv

# ============================================================

import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import polars as pl
except Exception:
    pl = None

warnings.filterwarnings("ignore")


# ============================================================
# 0. CONFIG
# ============================================================

current_path = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in [current_path, *current_path.parents]
     if (p / "Scripts").is_dir()),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from within the repository.")

BRIGHTDATA_DIR = PROJECT_ROOT / "Data" / "BrightData_Lawyers"
CROSSWALKS_DIR = PROJECT_ROOT / "Data" / "Geography" / "Crosswalks"

# The original BrightData files are local source data and must NOT be
# committed to the public repository. For the local preprocessing run,
# this assumes they are temporarily present in Data/BrightData_Lawyers/.
RAW_BRIGHTDATA_DIR = BRIGHTDATA_DIR

raw_lawyer_files = [
    RAW_BRIGHTDATA_DIR / "snap_mi504g7pxmrn977ah.1.csv",
    RAW_BRIGHTDATA_DIR / "snap_mi504g7pxmrn977ah.2.csv",
    RAW_BRIGHTDATA_DIR / "snap_mi504g7pxmrn977ah.3.csv",
]

anonymized_lawyer_files = [
    BRIGHTDATA_DIR / "BrightData_Lawyers_Anonymized_1.csv",
    BRIGHTDATA_DIR / "BrightData_Lawyers_Anonymized_2.csv",
    BRIGHTDATA_DIR / "BrightData_Lawyers_Anonymized_3.csv",
]

zip_cbsa_path = CROSSWALKS_DIR / "ZIP_CBSA_122024.xlsx"
qcew_crosswalk_path = (
    CROSSWALKS_DIR / "qcew-county-msa-csa-crosswalk-clean.xlsx"
)

FIRM_COUNTS_OUT = BRIGHTDATA_DIR / "firm_address_counts_by_MSA.csv"

# Use the same corrected ZIP extraction as the complete pipeline.
USE_CORRECTED_ZIP_EXTRACTION = True

# Puerto Rico metropolitan CBSA codes removed from the MSA list.
PR_CBSAS = {41980, 38660, 32420, 25020, 11640, 10380}

# IDs start at 1,000,000 so every published ID is exactly seven digits.
ANON_ID_START = 1_000_000


# ============================================================
# 1. PRIVACY / COLUMN POLICY
# ============================================================

# These are the only seven fields allowed in the public row-level files.
PUBLIC_COLUMNS = [
    "ID",
    "zip_code",
    "mailing_zip",
    "city",
    "state",
    "specializations",
    "number_of_specializations",
]

# Source fields that are used temporarily to construct the public fields.
# Full addresses and the URL are never written to the anonymized outputs.
SOURCE_COLUMNS_NEEDED = [
    "url",
    "mailing_address",
    "address",
    "location",
    "areas_of_practice",
    "practice_count",
]

# Every other BrightData field is intentionally excluded. The reason for
# each exclusion is documented here so the privacy transformation is auditable.
REMOVAL_REASONS = {
    "admission": "Professional admission history can help re-identify a lawyer and is not needed for the analysis.",
    "isln": "External/profile identifier that can be linked back to a person; not needed after pseudonymization.",
    "law_school_attended": "Education history is potentially identifying and is not used in the analysis.",
    "name": "Direct personal identifier.",
    "type": "Profile-type metadata is not used in the lawyer-count or specialty analysis.",
    "university_attended": "Education history is potentially identifying and is not used in the analysis.",
    "year_of_first_admission": "Career-history detail can help re-identify a lawyer and is not needed.",
    "filial": "Office/location text is unnecessary after retaining only ZIP/city/state geography.",
    "people": "Nested people/profile information may contain identifying information and is not used.",
    "awards": "Individual career distinctions can help re-identify a lawyer and are not used.",
    "profile_peer_review_count": "Peer-review metadata is not used in the analysis.",
    "profile_peer_review_star": "Peer-review rating is not used in the analysis.",
    "profile_peer_review_awards": "Peer-review distinctions can help re-identify a lawyer and are not used.",
    "fax": "Direct contact information.",
    "languages": "Personal/professional descriptor is not required for the analysis.",
    "office_hours": "Office metadata is not used in the analysis.",
    "office_size": "Office metadata is not used in the analysis.",
    "phone": "Direct contact information.",
    "photo": "Direct personal identifier.",
    "profile_peer_review_detail": "Detailed review information is not used and may contain identifying text.",
    "profile_visibility": "Profile-platform metadata is not used in the analysis.",
    "video_call": "Contact/availability metadata is not used in the analysis.",
    "website": "Directly linkable contact/profile information.",
    "biography": "Free-text biography can contain extensive identifying information.",
    "birth_information": "Personal biographical information is identifying and not needed.",
    "memberships": "Professional memberships can help re-identify a lawyer and are not needed.",
    "hobbies_interests": "Personal information unrelated to the analysis.",
    "profile_client_recomendation_count": "Client-review metadata is not used in the analysis.",
    "profile_client_recomendation_rating": "Client-review metadata is not used in the analysis.",
    "profile_client_review_count": "Client-review metadata is not used in the analysis.",
    "profile_client_review_detail": "Detailed client-review text is not used and may contain identifying information.",
    "profile_client_review_list": "Client-review text/list data is not used and may contain identifying information.",
    "profile_client_review_rating": "Client-review metadata is not used in the analysis.",
    "clients": "Client associations may be identifying and are not needed.",
    "clients2": "Client associations may be identifying and are not needed.",
    "year_established": "Office/business history is not used in the analysis.",
    "about": "Free-text profile information can contain identifying details.",
    "payment_information": "Payment/business information is not used in the analysis.",
    "state_bar_summary": "Professional profile detail can be identifying and is not needed.",
    "transactions": "Transaction/profile data are not used in the analysis.",
    "minority_owned": "Business demographic attribute is not needed for the analysis.",
    "phone_cell": "Direct contact information.",
    "phone_telecopier": "Direct contact information.",
    "company": "Employer/firm identity is not needed in the public row-level data.",
}

# The following source fields are transformed rather than directly published:
# - url -> ID: replace the linkable profile URL with a seven-digit code.
# - mailing_address -> mailing_zip: keep only the ZIP, then discard the address.
# - mailing_address/address/location -> zip_code: keep only the final analysis ZIP.
# - areas_of_practice -> specializations: retain the practice labels needed by the master pipeline.
# - practice_count -> number_of_specializations: retain only the count.
# City and state are derived from the HUD ZIP file, so no street-level address
# or raw office-location string is needed in the public data.


# ============================================================
# 2. HELPERS
# ============================================================

def print_header(title):
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)


def require_file(path, label):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing {label}: {path}")
    return path


def ensure_output_dir():
    BRIGHTDATA_DIR.mkdir(parents=True, exist_ok=True)


def read_csv_robust(path, usecols=None):
    path = str(path)
    if pl is not None:
        try:
            if usecols is not None:
                return pl.read_csv(
                    path,
                    columns=usecols,
                    ignore_errors=True,
                ).to_pandas()
            return pl.read_csv(path, ignore_errors=True).to_pandas()
        except Exception:
            pass
    return pd.read_csv(
        path,
        usecols=usecols,
        low_memory=False,
    )


ZIP_RE = r"(?<!\d)(\d{5})(?:-\d{4})?(?!\d)"


def extract_real_zip_from_text(txt):
    """
    Use the rightmost ZIP-like value because an address can contain a
    five-digit street or P.O. box number before its ZIP.
    """
    if not isinstance(txt, str):
        return None
    matches = re.findall(ZIP_RE, txt)
    return matches[-1] if matches else None


def extract_first_zip_from_text(txt):
    """Older first-five-digit behavior retained for compatibility."""
    if not isinstance(txt, str):
        return None
    match = re.search(r"\b\d{5}\b", txt)
    return match.group(0) if match else None


def extract_zip_from_row(row):
    # This is the same priority used by the current master pipeline:
    # mailing address first, then address, then location.
    for field in ["mailing_address", "address", "location"]:
        if field in row.index:
            txt = row.get(field)
            if USE_CORRECTED_ZIP_EXTRACTION:
                value = extract_real_zip_from_text(txt)
            else:
                value = extract_first_zip_from_text(txt)
            if value is not None:
                return value
    return None


def make_lawyer_id(df, path):
    """
    Reproduce the current pipeline's internal identity before anonymization:
    use URL when available, otherwise file path + row number.
    """
    fallback_ids = pd.Series(
        [f"{path}__{i}" for i in range(len(df))],
        index=df.index,
    )

    if "url" not in df.columns:
        return fallback_ids.astype(str)

    url_ids = df["url"].astype("string").str.strip()
    bad_url = url_ids.isna() | url_ids.str.lower().isin(
        ["", "nan", "none", "null", "na"]
    )
    return url_ids.where(~bad_url, fallback_ids).astype(str)


def check_raw_headers():
    """
    Confirm that every raw field is either transformed for publication or
    explicitly documented as excluded.
    """
    print_header("CHECKING RAW COLUMN PRIVACY POLICY")

    transformed = set(SOURCE_COLUMNS_NEEDED)
    documented_removed = set(REMOVAL_REASONS)

    for path in raw_lawyer_files:
        require_file(path, "original BrightData lawyer CSV")
        columns = set(pd.read_csv(path, nrows=0).columns)

        unknown = columns - transformed - documented_removed
        if unknown:
            raise ValueError(
                f"Undocumented raw columns in {path.name}: {sorted(unknown)}"
            )

        print(f"✓ {path.name}: {len(columns)} raw columns accounted for")

    print("Only these public columns will be written:")
    print(PUBLIC_COLUMNS)


# ============================================================
# 3. MSA / ZIP REFERENCE TABLES
# ============================================================

def build_msa_reference_tables():
    print("\n[1/5] Building valid MSA and CBSA-name tables...")

    require_file(qcew_crosswalk_path, "QCEW county-to-MSA crosswalk")
    crosswalk = pd.read_excel(qcew_crosswalk_path, dtype=str)
    crosswalk = crosswalk[
        crosswalk["MSA Title"].str.endswith(" MSA", na=False)
    ].copy()

    # Preserve the original pipeline's CBSA conversion logic.
    crosswalk["CBSA"] = (
        crosswalk["MSA Code"]
        .astype(str)
        .str[-4:]
        .str.ljust(5, "0")
        .astype(int)
    )

    valid_msa_codes = set(crosswalk["CBSA"].unique()) - PR_CBSAS

    cbsa_title = (
        crosswalk[["CBSA", "MSA Title"]]
        .drop_duplicates()
        .rename(columns={"MSA Title": "CBSA_Name"})
    )
    cbsa_title = cbsa_title[
        cbsa_title["CBSA"].isin(valid_msa_codes)
    ].copy()
    cbsa_title["CBSA_Name"] = (
        cbsa_title["CBSA_Name"]
        .str.replace(" MSA", "", regex=False)
        .str.replace(
            "Louisville-Jefferson County",
            "Louisville/Jefferson County",
            regex=False,
        )
    )

    print("   Valid MSA codes after Puerto Rico removal:", len(valid_msa_codes))
    print("   MSA names retained:", len(cbsa_title))
    return valid_msa_codes, cbsa_title


def load_zip_cbsa_fixed(valid_msa_codes):
    print("\n[2/5] Loading ZIP-to-CBSA crosswalk...")

    require_file(zip_cbsa_path, "ZIP-to-CBSA crosswalk")
    raw = pd.read_excel(zip_cbsa_path)
    raw["ZIP"] = raw["ZIP"].astype(str).str.zfill(5)
    raw["CBSA"] = pd.to_numeric(raw["CBSA"], errors="coerce")
    raw = raw.dropna(subset=["ZIP", "CBSA"])
    raw["CBSA"] = raw["CBSA"].astype(int)

    raw["is_valid_msa"] = raw["CBSA"].isin(valid_msa_codes).astype(int)
    raw["is_99999"] = (raw["CBSA"] == 99999).astype(int)

    ratio_priority = []
    for column in ["BUS_RATIO", "TOT_RATIO", "RES_RATIO", "OTH_RATIO"]:
        if column in raw.columns:
            raw[column] = pd.to_numeric(
                raw[column], errors="coerce"
            ).fillna(0)
            ratio_priority.append(column)

    sort_cols = (
        ["ZIP", "is_valid_msa", "is_99999"]
        + ratio_priority
        + ["CBSA"]
    )
    ascending = (
        [True, False, True]
        + [False] * len(ratio_priority)
        + [True]
    )

    zip_to_cbsa = (
        raw.sort_values(sort_cols, ascending=ascending)
        .drop_duplicates(subset="ZIP", keep="first")
        [["ZIP", "CBSA", "is_valid_msa"] + ratio_priority]
        .copy()
    )

    print("   ZIP-to-CBSA rows:", f"{len(zip_to_cbsa):,}")
    print(
        "   ZIPs assigned to valid metro MSA:",
        f"{int(zip_to_cbsa['is_valid_msa'].sum()):,}",
    )
    return zip_to_cbsa


def load_zip_city_state():
    """
    Derive non-street-level city/state geography from the HUD ZIP file.
    These fields are informational only and are not used to build the master.
    """
    raw = pd.read_excel(zip_cbsa_path, dtype={"ZIP": str})
    raw["ZIP"] = raw["ZIP"].astype(str).str.zfill(5)

    city_col = "USPS_ZIP_PREF_CITY"
    state_col = "USPS_ZIP_PREF_STATE"

    if city_col not in raw.columns or state_col not in raw.columns:
        raise ValueError(
            "ZIP_CBSA_122024.xlsx must contain USPS_ZIP_PREF_CITY and "
            "USPS_ZIP_PREF_STATE to create city/state fields."
        )

    geo = (
        raw[["ZIP", city_col, state_col]]
        .drop_duplicates(subset=["ZIP"], keep="first")
        .rename(columns={
            city_col: "city",
            state_col: "state",
        })
    )
    return (
        dict(zip(geo["ZIP"], geo["city"])),
        dict(zip(geo["ZIP"], geo["state"])),
    )


# ============================================================
# 4. LAWYER GEOGRAPHY NEEDED FOR FIRM COUNTS
# ============================================================

def load_lawyer_geography():
    print("\n[3/5] Loading lawyer IDs and analysis ZIPs...")

    chunks = []
    raw_row_count = 0
    usecols = ["url", "mailing_address", "address", "location"]

    for path in raw_lawyer_files:
        print("   Loading:", path)
        df = read_csv_robust(path, usecols=usecols)
        raw_row_count += len(df)
        df["lawyer_id"] = make_lawyer_id(df, path)
        df["zip"] = df.apply(extract_zip_from_row, axis=1)
        chunks.append(df[["lawyer_id", "zip"]])

    all_lawyers = (
        pd.concat(chunks, ignore_index=True)
        .drop_duplicates(subset=["lawyer_id"])
        .reset_index(drop=True)
    )

    print("   Raw rows loaded:", f"{raw_row_count:,}")
    print(
        "   Unique lawyers retained:",
        f"{all_lawyers['lawyer_id'].nunique():,}",
    )
    return all_lawyers


def assign_msa_to_lawyers(
    all_lawyers,
    zip_to_cbsa,
    valid_msa_codes,
    cbsa_title,
):
    # This is the same MSA assignment used by the current master pipeline.
    lawyer_zip_rows = all_lawyers[["lawyer_id", "zip"]].drop_duplicates(
        subset=["lawyer_id", "zip"]
    )
    lawyer_zip_rows["zip"] = lawyer_zip_rows["zip"].astype("string")

    msa_rows = lawyer_zip_rows.merge(
        zip_to_cbsa,
        left_on="zip",
        right_on="ZIP",
        how="left",
    )
    msa_rows["CBSA"] = pd.to_numeric(msa_rows["CBSA"], errors="coerce")

    msa_rows["msa_status"] = "unknown"
    msa_rows.loc[msa_rows["zip"].isna(), "msa_status"] = "no_zip_extracted"
    msa_rows.loc[
        msa_rows["zip"].notna() & msa_rows["CBSA"].isna(),
        "msa_status",
    ] = "zip_not_in_zip_cbsa_crosswalk"
    msa_rows.loc[
        msa_rows["CBSA"].isin(PR_CBSAS),
        "msa_status",
    ] = "puerto_rico_removed"
    msa_rows.loc[
        msa_rows["CBSA"].notna()
        & ~msa_rows["CBSA"].isin(valid_msa_codes)
        & ~msa_rows["CBSA"].isin(PR_CBSAS),
        "msa_status",
    ] = "cbsa_not_valid_metro_msa"
    msa_rows.loc[
        msa_rows["CBSA"].isin(valid_msa_codes),
        "msa_status",
    ] = "valid_msa"

    valid_rows = msa_rows[msa_rows["msa_status"] == "valid_msa"].copy()

    if len(valid_rows) > 0:
        valid_rows["CBSA"] = valid_rows["CBSA"].astype(int)
        valid_rows = valid_rows.merge(
            cbsa_title.rename(columns={"CBSA_Name": "MSA"}),
            on="CBSA",
            how="left",
        )

        msa_per_lawyer = (
            valid_rows.groupby(["lawyer_id", "CBSA", "MSA"])
            .size()
            .reset_index(name="n_zip_rows")
            .sort_values(
                ["lawyer_id", "n_zip_rows", "CBSA"],
                ascending=[True, False, True],
            )
            .drop_duplicates(subset="lawyer_id", keep="first")
            [["lawyer_id", "CBSA", "MSA"]]
        )
    else:
        msa_per_lawyer = pd.DataFrame(
            columns=["lawyer_id", "CBSA", "MSA"]
        )

    print(
        "   Lawyers with valid MSA:",
        f"{msa_per_lawyer['lawyer_id'].nunique():,}",
    )
    return msa_per_lawyer


# ============================================================
# 5. FIRM-ADDRESS OUTPUT
# ============================================================
#
# This section intentionally reproduces the firm-address logic from
# Lawyer_Paper_Complete_Pipeline.ipynb before any street address is removed.

def choose_address(row):
    for column in ["mailing_address", "address", "location"]:
        if column in row.index:
            value = row.get(column)
            if (
                pd.notna(value)
                and str(value).strip().lower()
                not in ["", "nan", "none", "null", "na"]
            ):
                return str(value)
    return None


def clean_address_for_grouping(value):
    if pd.isna(value):
        return pd.NA
    value = str(value).lower().strip()
    value = re.sub(r"\s+", " ", value)
    value = re.sub(r"[^\w\s#/-]", "", value)
    return value.strip()


def build_and_save_firm_counts(
    zip_to_cbsa,
    msa_per_lawyer,
):
    """
    Build the firm-address counts.
    A firm address is an address shared by at least two unique lawyers.
    """
    print_header("BUILDING FIRM-ADDRESS COUNTS")

    address_chunks = []

    for path in raw_lawyer_files:
        print("Loading addresses from:", path)
        tmp = read_csv_robust(path)
        tmp["lawyer_id"] = make_lawyer_id(tmp, path)
        tmp["zip"] = tmp.apply(extract_zip_from_row, axis=1)
        tmp["address_raw"] = tmp.apply(choose_address, axis=1)
        tmp["address_key"] = tmp["address_raw"].map(
            clean_address_for_grouping
        )
        address_chunks.append(tmp[["lawyer_id", "zip", "address_key"]])

    lawyer_addresses = pd.concat(address_chunks, ignore_index=True)
    lawyer_addresses = lawyer_addresses.dropna(
        subset=["lawyer_id", "zip", "address_key"]
    )
    lawyer_addresses = lawyer_addresses.drop_duplicates(
        subset=["lawyer_id", "zip", "address_key"]
    )

    lawyer_addresses = lawyer_addresses.merge(
        zip_to_cbsa[["ZIP", "CBSA"]],
        left_on="zip",
        right_on="ZIP",
        how="left",
    )
    lawyer_addresses["CBSA"] = pd.to_numeric(
        lawyer_addresses["CBSA"],
        errors="coerce",
    )
    lawyer_addresses = lawyer_addresses.dropna(subset=["CBSA"])
    lawyer_addresses["CBSA"] = lawyer_addresses["CBSA"].astype(int)

    lawyer_addresses = lawyer_addresses.merge(
        msa_per_lawyer[["lawyer_id", "CBSA", "MSA"]],
        on=["lawyer_id", "CBSA"],
        how="inner",
    )

    lawyer_addresses = (
        lawyer_addresses.sort_values(["MSA", "lawyer_id", "address_key"])
        .drop_duplicates(subset=["lawyer_id"], keep="first")
    )

    address_counts = (
        lawyer_addresses.groupby(["CBSA", "MSA", "address_key"])
        .agg(
            lawyers_at_address=("lawyer_id", "nunique"),
        )
        .reset_index()
    )

    address_counts["is_firm_address"] = (
        address_counts["lawyers_at_address"] >= 2
    )
    address_counts["is_solo_address"] = (
        address_counts["lawyers_at_address"] == 1
    )

    firm_address_stats = (
        address_counts.groupby(["CBSA", "MSA"])
        .agg(
            firm_addresses=("is_firm_address", lambda x: int(x.sum())),
            solo_addresses=("is_solo_address", lambda x: int(x.sum())),
            lawyers_in_firms=(
                "lawyers_at_address",
                lambda x: int(x[x >= 2].sum()),
            ),
            lawyers_solo=(
                "lawyers_at_address",
                lambda x: int(x[x == 1].sum()),
            ),
            total_addresses=("address_key", "nunique"),
        )
        .reset_index()
    )

    firm_address_stats["pct_addresses_firms"] = (
        firm_address_stats["firm_addresses"]
        / firm_address_stats["total_addresses"]
    )
    firm_address_stats["pct_addresses_solo"] = (
        firm_address_stats["solo_addresses"]
        / firm_address_stats["total_addresses"]
    )

    firm_address_stats.to_csv(FIRM_COUNTS_OUT, index=False)
    print(f"✓ Saved firm address counts: {FIRM_COUNTS_OUT}")

    return firm_address_stats


# ============================================================
# 6. ANONYMIZATION
# ============================================================

def build_anonymous_id_map(all_lawyers):
    """
    Assign one seven-digit pseudonymous ID to each unique internal lawyer ID.
    Repeated occurrences of the same source lawyer receive the same public ID.
    The source-to-public mapping is kept only in memory and is never written.
    """
    unique_ids = all_lawyers["lawyer_id"].drop_duplicates().tolist()

    last_id = ANON_ID_START + len(unique_ids) - 1
    if last_id > 9_999_999:
        raise ValueError(
            "There are too many unique lawyers to represent all IDs with "
            "seven digits."
        )

    return dict(
        zip(
            unique_ids,
            range(ANON_ID_START, ANON_ID_START + len(unique_ids)),
        )
    )


def specialization_count(series):
    """
    Fallback count used only when BrightData practice_count is missing.
    It counts the comma-separated source practice labels.
    """
    def count_one(value):
        if pd.isna(value):
            return 0
        parts = [part.strip() for part in str(value).split(",")]
        return sum(bool(part) for part in parts)

    return series.map(count_one).astype("Int64")


def write_anonymized_files(
    anonymous_id_map,
    zip_to_city,
    zip_to_state,
):
    print_header("WRITING ANONYMIZED BRIGHTDATA FILES")

    usecols = [
        "url",
        "mailing_address",
        "address",
        "location",
        "areas_of_practice",
        "practice_count",
    ]

    for raw_path, output_path in zip(
        raw_lawyer_files,
        anonymized_lawyer_files,
    ):
        print("Anonymizing:", raw_path.name)

        df = read_csv_robust(raw_path, usecols=usecols)
        df["lawyer_id"] = make_lawyer_id(df, raw_path)

        # Replace the linkable URL/source identifier with a seven-digit code.
        df["ID"] = df["lawyer_id"].map(anonymous_id_map)
        if df["ID"].isna().any():
            raise ValueError(f"Missing anonymized IDs in {raw_path.name}")
        df["ID"] = df["ID"].astype(int)

        # Keep the exact ZIP used by the master pipeline, but no street address.
        df["zip_code"] = df.apply(extract_zip_from_row, axis=1)
        df["zip_code"] = df["zip_code"].astype("string")

        # Retain mailing ZIP separately as requested, without mailing address.
        df["mailing_zip"] = df["mailing_address"].map(
            extract_real_zip_from_text
        )
        df["mailing_zip"] = df["mailing_zip"].astype("string")

        # City/state are derived from the published HUD ZIP geography rather
        # than retaining potentially identifying raw office-location text.
        df["city"] = df["zip_code"].map(zip_to_city)
        df["state"] = df["zip_code"].map(zip_to_state)

        # Preserve the source practice-area text needed for the 13-category
        # crosswalk, but publish it under the neutral field name requested.
        df["specializations"] = df["areas_of_practice"]

        # Keep only the number of source practice areas, not any other profile
        # metadata. Use BrightData's count when available.
        counts = pd.to_numeric(
            df["practice_count"],
            errors="coerce",
        ).astype("Int64")
        missing_count = counts.isna()
        if missing_count.any():
            counts.loc[missing_count] = specialization_count(
                df.loc[missing_count, "areas_of_practice"]
            )
        df["number_of_specializations"] = counts

        public = df[PUBLIC_COLUMNS].copy()

        # Hard privacy guard: the output cannot contain any undeclared column.
        if list(public.columns) != PUBLIC_COLUMNS:
            raise ValueError("Unexpected columns in anonymized output.")

        public.to_csv(output_path, index=False)

        print(
            f"✓ Saved {output_path.name}: "
            f"{len(public):,} rows, {len(public.columns)} columns"
        )

    return anonymized_lawyer_files


# ============================================================
# 7. MAIN
# ============================================================

def main():
    ensure_output_dir()
    print_header("STARTING PREPROCESSING AND ANONYMIZATION")

    check_raw_headers()

    valid_msa_codes, cbsa_title = build_msa_reference_tables()
    zip_to_cbsa = load_zip_cbsa_fixed(valid_msa_codes)

    all_lawyers = load_lawyer_geography()
    msa_per_lawyer = assign_msa_to_lawyers(
        all_lawyers,
        zip_to_cbsa,
        valid_msa_codes,
        cbsa_title,
    )

    # Firm counts must be built before addresses are removed.
    firm_counts = build_and_save_firm_counts(
        zip_to_cbsa,
        msa_per_lawyer,
    )

    # Create anonymous row-level files only after the aggregate firm file exists.
    anonymous_id_map = build_anonymous_id_map(all_lawyers)
    zip_to_city, zip_to_state = load_zip_city_state()
    anonymized_files = write_anonymized_files(
        anonymous_id_map,
        zip_to_city,
        zip_to_state,
    )

    print_header("PREPROCESSING COMPLETE")
    print("Public/repository outputs:")
    print("  ", FIRM_COUNTS_OUT)
    for path in anonymized_files:
        print("  ", path)

    print("\nDo NOT upload the original snap_mi504g7pxmrn977ah.*.csv files.")

    return firm_counts, anonymized_files


if __name__ == "__main__":
    firm_counts, anonymized_files = main()



STARTING PREPROCESSING AND ANONYMIZATION

CHECKING RAW COLUMN PRIVACY POLICY
✓ snap_mi504g7pxmrn977ah.1.csv: 50 raw columns accounted for
✓ snap_mi504g7pxmrn977ah.2.csv: 50 raw columns accounted for
✓ snap_mi504g7pxmrn977ah.3.csv: 50 raw columns accounted for
Only these public columns will be written:
['ID', 'zip_code', 'mailing_zip', 'city', 'state', 'specializations', 'number_of_specializations']

[1/5] Building valid MSA and CBSA-name tables...
   Valid MSA codes after Puerto Rico removal: 387
   MSA names retained: 387

[2/5] Loading ZIP-to-CBSA crosswalk...
   ZIP-to-CBSA rows: 39,487
   ZIPs assigned to valid metro MSA: 25,020

[3/5] Loading lawyer IDs and analysis ZIPs...
   Loading: /Users/maxbelykh/Downloads/The-Urban-Legal-Mirage-main 3/Data/BrightData_Lawyers/snap_mi504g7pxmrn977ah.1.csv
   Loading: /Users/maxbelykh/Downloads/The-Urban-Legal-Mirage-main 3/Data/BrightData_Lawyers/snap_mi504g7pxmrn977ah.2.csv
   Loading: /Users/maxbelykh/Downloads/The-Urban-Legal-Mirage-main 

In [3]:
# ORIGINAL FILE STRUCTURE SUMMARY

# Display only the original column names and number of rows.
# No row-level or personally identifying data are printed.

print_header("ORIGINAL BRIGHTDATA FILE STRUCTURE")

for raw_path, anon_path in zip(raw_lawyer_files, anonymized_lawyer_files):
    
    # Read only the header from the original file.
    original_columns = pd.read_csv(raw_path, nrows=0).columns.tolist()
    
    # The anonymized files preserve one row for every original row,
    # so their row counts are identical to the source files.
    row_count = sum(1 for _ in open(anon_path, "rb")) - 1

    print(f"\nFile: {raw_path.name}")
    print(f"Number of rows: {row_count:,}")
    print(f"Number of columns: {len(original_columns)}")
    print("Original column names:")
    
    for column in original_columns:
        print(f"  - {column}")


ORIGINAL BRIGHTDATA FILE STRUCTURE

File: snap_mi504g7pxmrn977ah.1.csv
Number of rows: 865,932
Number of columns: 50
Original column names:
  - url
  - address
  - admission
  - areas_of_practice
  - isln
  - law_school_attended
  - location
  - name
  - practice_count
  - type
  - university_attended
  - year_of_first_admission
  - filial
  - people
  - awards
  - profile_peer_review_count
  - profile_peer_review_star
  - profile_peer_review_awards
  - fax
  - languages
  - mailing_address
  - office_hours
  - office_size
  - phone
  - photo
  - profile_peer_review_detail
  - profile_visibility
  - video_call
  - website
  - biography
  - birth_information
  - memberships
  - hobbies_interests
  - profile_client_recomendation_count
  - profile_client_recomendation_rating
  - profile_client_review_count
  - profile_client_review_detail
  - profile_client_review_list
  - profile_client_review_rating
  - clients
  - clients2
  - year_established
  - about
  - payment_information
  - sta